In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Connect and Access Git Repository

import sys
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"



token = getpass("GitHub token: ")


REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())


In [ ]:
!pip install -q shap

In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import shap
from torch.utils.data import DataLoader



# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.seed import set_seed, RANDOM_SEED
from models.tranad import TranADMainModel
from utils.dataset import PTBXLTranADDataset
from explainability.anomaly_score_wrappers import TranADAnomalyScoreWrapper
from explainability.shap_explainer import aggregate_shap_intervals, plot_normal_anomalous_shap_heatmaps
from explainability.evaluation import plot_temporal_importance


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

In [ ]:
# Paths

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models" / "tranad"

MODEL_PATH = MODEL_DIR / "tranad_20epochs_larger_capacity.pt"

SHAP_RESULTS_DIR = RESULTS_DIR / "shap"
SHAP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXPLANATION_SAMPLES_PATH = RESULTS_DIR / "evaluation" / "shared_explanation_samples.npz"


print("MODEL_PATH:", MODEL_PATH)
print("SHAP results:", SHAP_RESULTS_DIR)
print("Shared explanation samples:", EXPLANATION_SAMPLES_PATH)

In [ ]:
# Load my preprocessed data

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_val = np.load(PROCESSED_DIR / "X_val.npy")
y_val = np.load(PROCESSED_DIR / "y_val.npy")

X_test = np.load(PROCESSED_DIR / "X_test.npy")
y_test = np.load(PROCESSED_DIR / "y_test.npy")

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

In [ ]:
# Select normal data only

normal_train_mask = y_train == 0

X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

print("Training on normal ECGs:", X_train_normal.shape)

In [ ]:
# Datasets

train_dataset = PTBXLTranADDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)

val_dataset = PTBXLTranADDataset(
    X_val,
    y_val,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

test_dataset = PTBXLTranADDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

In [ ]:
# Dataloaders

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

In [ ]:
# Load the pre-trained model

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}"
    )

TranAD.Load(str(MODEL_PATH))
TranAD.Generator.to(DEVICE)
TranAD.Generator.eval()

print("TranAD checkpoint loaded.")

In [ ]:
ScoreWrapper = TranADAnomalyScoreWrapper(TranAD)

In [ ]:
background_indices = np.load(SHAP_RESULTS_DIR / "medium_background_indices.npy")

In [ ]:
# select ECG samples to explain with SHAP
# I choose the same two samples used for Attention and SHAP on AnoFormer

SharedSamples = np.load(
    RESULTS_DIR
    / "evaluation"
    / "shared_explanation_samples.npz"
)

NormalIndex = int(
    SharedSamples["normal_index"]
)

AnomalousIndex = int(
    SharedSamples["anomalous_index"]
)

NormalECG, NormalLabel = test_dataset[NormalIndex]
AnomalousECG, AnomalousLabel = test_dataset[AnomalousIndex]

NormalECG = NormalECG.unsqueeze(0).to(TranAD.Device)
AnomalousECG = AnomalousECG.unsqueeze(0).to(TranAD.Device)

print("Index of Normal ECG:", NormalIndex, "Label:", NormalLabel.item())
print("Index of Anomalous ECG:", AnomalousIndex, "Label:", AnomalousLabel.item())
print("Anomalous index:", AnomalousIndex)

In [ ]:
sample_indices = np.array(
    [NormalIndex, AnomalousIndex]
)

sample_ECGs = torch.as_tensor(
    X_test[sample_indices],
    dtype=torch.float32,
    device=DEVICE,
)

sample_Labels = y_test[sample_indices]

print("ECG shape:", sample_ECGs.shape)
print("Labels:", sample_Labels)

In [ ]:
# create a GradientSHAPEplainer and calculate the SHAP values

GradientSHAPExplainer = shap.GradientExplainer(
    ScoreWrapper,
    background_ECGs,
)

SHAPValues = GradientSHAPExplainer.shap_values(
    sample_ECGs
)

In [ ]:
# Remove the single-output dimension if it exists:
# (2, 1000, 12, 1) -> (2, 1000, 12)

if SHAPValues.ndim == 4 and SHAPValues.shape[-1] == 1:
    SHAPValues = SHAPValues[..., 0]

print("SHAP shape:", SHAPValues.shape)

In [ ]:
print("Contains NaN:", np.isnan(SHAPValues).any())
print("Contains infinity:", np.isinf(SHAPValues).any())
print("All values zero:", np.allclose(SHAPValues, 0))

print("Minimum:", SHAPValues.min())
print("Maximum:", SHAPValues.max())
print("Mean absolute value:", np.abs(SHAPValues).mean())

In [ ]:
# aggregate the values for better plots

NormalSHAPIntervals = aggregate_shap_intervals(
    SHAPValues[0],
    block_size=10,
    absolute=True,
)

AnomalousSHAPIntervals = aggregate_shap_intervals(
    SHAPValues[1],
    block_size=10,
    absolute=True,
)

print(NormalSHAPIntervals.shape)
print(AnomalousSHAPIntervals.shape)

In [ ]:
# Plot SHAP Heatmaps

LEAD_NAMES = [
    "I", "II", "III",
    "aVR", "aVL", "aVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6",
]

plot_normal_anomalous_shap_heatmaps(
    normal_values=NormalSHAPIntervals,
    anomalous_values=AnomalousSHAPIntervals,
    lead_names=LEAD_NAMES,
    interval_seconds=0.1,
    percentile=99.0,
    save_path=(
        SHAP_RESULTS_DIR
        / "tranad"
        / "shap_normal_vs_anomalous.png"
    ),
)

In [ ]:
NormalTemporalSHAP = NormalSHAPIntervals.mean(axis=1)
AnomalousTemporalSHAP = AnomalousSHAPIntervals.mean(axis=1)

print(NormalTemporalSHAP.shape)
print(AnomalousTemporalSHAP.shape)

In [ ]:
# Plot that is comparable with the Attention plot

plot_temporal_importance(
    values=NormalTemporalSHAP,
    title="TranAD SHAP importance — normal ECG",
    save_path=(
        SHAP_RESULTS_DIR
        / "tranad"
        / "temporal_shap_normal.png"
    ),
)

plot_temporal_importance(
    values=AnomalousTemporalSHAP,
    title="TranAD SHAP importance — anomalous ECG",
    save_path=(
        SHAP_RESULTS_DIR
        / "tranad"
        / "temporal_shap_anomalous.png"
    ),
)